In [2]:
import numpy as np
import math
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler

# =========================================================
# 1. Data: 5D inputs and 1D outputs (Function 6)
# =========================================================

X_raw = np.array([
    [0.7281861 , 0.15469257, 0.73255167, 0.69399651, 0.05640131],
    [0.24238435, 0.84409997, 0.5778091 , 0.67902128, 0.50195289],
    [0.72952261, 0.7481062 , 0.67977464, 0.35655228, 0.67105368],
    [0.77062024, 0.11440374, 0.04677993, 0.64832428, 0.27354905],
    [0.6188123 , 0.33180214, 0.18728787, 0.75623847, 0.3288348 ],
    [0.78495809, 0.91068235, 0.7081201 , 0.95922543, 0.0049115 ],
    [0.14511079, 0.8966846 , 0.89632223, 0.72627154, 0.23627199],
    [0.94506907, 0.28845905, 0.97880576, 0.96165559, 0.59801594],
    [0.12572016, 0.86272469, 0.02854433, 0.24660527, 0.75120624],
    [0.75759436, 0.35583141, 0.0165229 , 0.4342072 , 0.11243304],
    [0.5367969 , 0.30878091, 0.41187929, 0.38822518, 0.5225283 ],
    [0.95773967, 0.23566857, 0.09914585, 0.15680593, 0.07131737],
    [0.6293079 , 0.80348368, 0.81140844, 0.04561319, 0.11062446],
    [0.02173531, 0.42808424, 0.83593944, 0.48948866, 0.51108173],
    [0.43934426, 0.69892383, 0.42682022, 0.10947609, 0.87788847],
    [0.25890557, 0.79367771, 0.6421139 , 0.19667346, 0.59310318],
    [0.43216593, 0.71561781, 0.3418191 , 0.70499988, 0.61496184],
    [0.78287982, 0.53633586, 0.44328356, 0.85969983, 0.01032599],
    [0.9217762 , 0.93187122, 0.41487637, 0.59505727, 0.73562569],
    [0.12667892, 0.2914703 , 0.06452848, 0.6805146 , 0.89281919],
    [1.057739  , 1.031871  , 1.078805  , 1.061655  , 0.992819  ],
    [0.183405  , 0.304243  , 0.524756  , 0.431945  , 0.29123   ],
    [0.268807  , 0.268756  , 0.495982  , 0.986904  , 0.010463  ],
    [0.071886  , 0.119564  , 0.11427   , 0.97486   , 0.062381  ],
])

y_raw = np.array([
    6.44434399e+01, 1.83013796e+01, 1.12939795e-01, 4.21089813e+00,
    2.58370525e+02, 7.84343889e+01, 5.75715369e+01, 1.09571876e+02,
    8.84799176e+00, 2.33223610e+02, 2.44230883e+01, 6.44201468e+01,
    6.34767158e+01, 7.97291299e+01, 3.55806818e+02, 1.08885962e+03,
    2.88667516e+01, 4.51815703e+01, 4.31612757e+02, 9.97233189e+00,
    7.71337361e+03, 1.61662575e+03, 5.63309324e+02, 3.46190426e+03
])

# =========================================================
# 2. Reproducibility and device
# =========================================================

RANDOM_SEED = 123
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# =========================================================
# 3. Scaling (critical for neural nets)
# =========================================================

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_scaled = x_scaler.fit_transform(X_raw)
y_scaled = y_scaler.fit_transform(y_raw.reshape(-1, 1)).ravel()

X_tensor = torch.tensor(X_scaled, dtype=torch.float32, device=device)
y_tensor = torch.tensor(y_scaled.reshape(-1, 1), dtype=torch.float32, device=device)

# =========================================================
# 4. Deep NN surrogate with dropout (for MC uncertainty)
# =========================================================

class SurrogateNN(nn.Module):
    def __init__(self, input_dim=5, hidden_dim=64, dropout_p=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return self.net(x)

model = SurrogateNN(input_dim=5, hidden_dim=64, dropout_p=0.1).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

# =========================================================
# 5. Train with simple early stopping
# =========================================================

def train_model(model, X, y, n_epochs=1500, patience=200):
    n_samples = X.shape[0]
    n_train = max(int(0.8 * n_samples), 1)

    indices = torch.randperm(n_samples)
    train_idx = indices[:n_train]
    val_idx = indices[n_train:]

    X_train, y_train = X[train_idx], y[train_idx]
    X_val, y_val = X[val_idx], y[val_idx]

    best_val_loss = float("inf")
    best_state = None
    epochs_no_improve = 0

    for epoch in range(1, n_epochs + 1):
        # --- training step ---
        model.train()
        optimizer.zero_grad()
        preds = model(X_train)
        loss = criterion(preds, y_train)
        loss.backward()
        optimizer.step()

        # --- validation step ---
        model.eval()
        with torch.no_grad():
            val_preds = model(X_val)
            val_loss = criterion(val_preds, y_val).item()

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1

        if epochs_no_improve >= patience:
            break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    return model

model = train_model(model, X_tensor, y_tensor)

# =========================================================
# 6. MC Dropout prediction in original y space
# =========================================================

def predict_mc(model, X, n_samples=50):
    """
    X: candidates in original 0–1 space, shape (N, 5)
    Returns: predictive mean and std in ORIGINAL y units (not scaled).
    """
    model.train()  # keep dropout active
    X_scaled = x_scaler.transform(X)
    X_t = torch.tensor(X_scaled, dtype=torch.float32, device=device)

    preds = []
    with torch.no_grad():
        for _ in range(n_samples):
            out = model(X_t)          # (N, 1)
            preds.append(out.cpu().numpy())

    preds = np.stack(preds, axis=0)  # (MC, N, 1)
    preds = preds.squeeze(-1)        # (MC, N)

    # inverse transform to original scale
    preds_flat = preds.reshape(-1, 1)
    preds_orig = y_scaler.inverse_transform(preds_flat).reshape(preds.shape)

    mean = preds_orig.mean(axis=0)
    std = preds_orig.std(axis=0)
    return mean, std

# =========================================================
# 7. Acquisition: Expected Improvement (EI) & Probability of Improvement (PI)
#    Implemented without SciPy using erf.
# =========================================================

def normal_pdf(z):
    return np.exp(-0.5 * z ** 2) / math.sqrt(2.0 * math.pi)

def normal_cdf(z):
    return 0.5 * (1.0 + math.erf(z / math.sqrt(2.0)))

def acquisition_ei_pi(mu, sigma, y_best, xi=0.01):
    """
    mu, sigma: arrays (N,) – predicted mean and std.
    y_best: scalar – current best observed y.
    xi: small positive number encouraging exploration.
    Returns: EI (N,), PI (N,)
    """
    eps = 1e-9
    sigma = np.maximum(sigma, eps)

    improvement = mu - y_best - xi
    Z = improvement / sigma

    # vectorized CDF/PDF
    cdf_vals = np.vectorize(normal_cdf)(Z)
    pdf_vals = np.vectorize(normal_pdf)(Z)

    ei = improvement * cdf_vals + sigma * pdf_vals
    pi = cdf_vals

    ei = np.maximum(ei, 0.0)
    return ei, pi

# =========================================================
# 8. Propose next query point inside [0,1]^5 (respecting bounds)
# =========================================================

def propose_next_point(
    model,
    X_observed,
    y_observed,
    n_candidates=8000,
    xi=0.01,
    mc_samples=50,
    random_seed=123,
):
    """
    Samples candidate points uniformly in [0,1]^5 and chooses the one
    that maximizes Expected Improvement over the current best observed y.
    """
    rng = np.random.default_rng(random_seed)

    # All candidate inputs live strictly within [0, 1]^5
    X_cand = rng.random((n_candidates, X_observed.shape[1]))

    mu, sigma = predict_mc(model, X_cand, n_samples=mc_samples)

    y_best = float(np.max(y_observed))
    ei, pi = acquisition_ei_pi(mu, sigma, y_best, xi=xi)

    best_idx = int(np.argmax(ei))
    x_next = X_cand[best_idx]
    mu_next = float(mu[best_idx])
    sigma_next = float(sigma[best_idx])
    pi_next = float(pi[best_idx])
    ei_next = float(ei[best_idx])

    best_obs_idx = int(np.argmax(y_observed))
    x_best_obs = X_observed[best_obs_idx]
    y_best_obs = float(y_observed[best_obs_idx])

    details = {
        "x_next": x_next,
        "mu_next": mu_next,
        "sigma_next": sigma_next,
        "pi_next": pi_next,
        "ei_next": ei_next,
        "x_best_obs": x_best_obs,
        "y_best_obs": y_best_obs,
    }
    return details

# =========================================================
# 9. Local sensitivity: gradient-based feature importance at x_next
# =========================================================

def local_sensitivity(model, x_point):
    """
    x_point: shape (5,), original space.
    Returns normalized absolute gradient magnitudes per dimension.
    """
    model.eval()
    x_scaled = x_scaler.transform(x_point.reshape(1, -1))
    x_t = torch.tensor(x_scaled, dtype=torch.float32, device=device, requires_grad=True)
    y_pred_scaled = model(x_t)
    y_pred_scaled.backward()

    grads = x_t.grad.detach().cpu().numpy().flatten()
    grads_abs = np.abs(grads)
    if grads_abs.sum() == 0:
        return np.ones_like(grads_abs) / len(grads_abs)
    importance = grads_abs / grads_abs.sum()
    return importance

# =========================================================
# 10. Main: train surrogate, propose next point, print reasoning
# =========================================================

def main():
    details = propose_next_point(
        model,
        X_raw,
        y_raw,
        n_candidates=8000,  # search resolution in [0,1]^5
        xi=0.01,
        mc_samples=50,
        random_seed=RANDOM_SEED,
    )

    x_next = details["x_next"]
    mu_next = details["mu_next"]
    sigma_next = details["sigma_next"]
    pi_next = details["pi_next"]
    ei_next = details["ei_next"]
    x_best_obs = details["x_best_obs"]
    y_best_obs = details["y_best_obs"]

    print("=== CURRENT BEST (from observed data) ===")
    print(f"Best observed input x*  : {x_best_obs}")
    print(f"Best observed output y*: {y_best_obs:.6f}")
    print()

    print("=== PROPOSED NEXT QUERY POINT (model-based) ===")
    print(f"Next query x_next (within [0,1]^5): {x_next}")
    print(f"Predicted mean output f(x_next): {mu_next:.4f}")
    print(f"Predictive std dev at x_next   : {sigma_next:.4f}")
    print(f"Expected Improvement (EI)      : {ei_next:.4f}")
    print(f"Probability of Improvement (PI): {pi_next:.4f}")
    print()

    # Local, gradient-based ingredient importance
    importance = local_sensitivity(model, x_next)
    print("Approximate local ingredient importance at x_next (normalized):")
    for i, imp in enumerate(importance, start=1):
        print(f"  Dimension {i}: {imp:.3f}")
    print()

    # Textual reasoning
    print("=== REASONING SUMMARY ===")
    print(
        "1) A deep neural network surrogate with dropout is trained on scaled "
        "inputs/outputs, turning the black-box cake score into a smooth model."
    )
    print(
        "2) Dropout is kept active at prediction time and multiple forward passes "
        "(MC Dropout) approximate a predictive distribution for each candidate recipe."
    )
    print(
        f"3) Among 8000 random candidates inside the allowed [0,1]^5 bounds, "
        "x_next maximizes the Expected Improvement acquisition function relative "
        f"to the current best observed score y* = {y_best_obs:.3f}."
    )
    print(
        f"4) At x_next the surrogate predicts an average score of {mu_next:.1f} "
        f"with uncertainty (std) {sigma_next:.1f}. The probability that this "
        f"recipe beats the current best (PI) is about {pi_next*100:.1f}%. "
        "High EI and PI justify selecting x_next as the next experiment."
    )
    print(
        "5) The gradient-based local importance values show which of the five "
        "ingredients drive the predicted score near x_next, giving an interpretable "
        "reason for why the model prefers this region."
    )

if __name__ == "__main__":
    main()


=== CURRENT BEST (from observed data) ===
Best observed input x*  : [1.057739 1.031871 1.078805 1.061655 0.992819]
Best observed output y*: 7713.373610

=== PROPOSED NEXT QUERY POINT (model-based) ===
Next query x_next (within [0,1]^5): [0.98505348 0.91285682 0.96772398 0.99302192 0.94406231]
Predicted mean output f(x_next): 5740.0361
Predictive std dev at x_next   : 615.6484
Expected Improvement (EI)      : 0.1118
Probability of Improvement (PI): 0.0007

Approximate local ingredient importance at x_next (normalized):
  Dimension 1: 0.151
  Dimension 2: 0.337
  Dimension 3: 0.117
  Dimension 4: 0.156
  Dimension 5: 0.239

=== REASONING SUMMARY ===
1) A deep neural network surrogate with dropout is trained on scaled inputs/outputs, turning the black-box cake score into a smooth model.
2) Dropout is kept active at prediction time and multiple forward passes (MC Dropout) approximate a predictive distribution for each candidate recipe.
3) Among 8000 random candidates inside the allowed [0,